# Day 11 — Final Model Pipeline, Model Persistence ve Streamlit Dashboard

## Hedefler

- Güncel model seçimlerini final konfigürasyon olarak sabitlemek
- Seçilen modelleri tüm güncel veri üzerinde eğitmek
- Eğitilmiş model nesnelerini `models/` klasörüne kaydetmek
- Kaydedilen modelleri tekrar yükleyerek doğrulamak
- Model konfigürasyonlarını metadata olarak saklamak
- Kaydedilmiş modellerle yeniden eğitim yapmadan tahmin üreten bir pipeline oluşturmak
- Final 4 haftalık forecast çıktısını kaydetmek
- Streamlit ve Plotly kullanarak interaktif bir dashboard geliştirmek
- Anomaly detection ve early-warning mekanizmalarını dashboard'a entegre etmek

In [1]:
# --------------------------------------------------
# Day 11 - Setup
# --------------------------------------------------

# pandas:
# CSV dosyasını okumak ve zaman serisi verileriyle
# çalışmak için kullanıyoruz.
import pandas as pd

# numpy:
# Sayısal işlemler ve gerektiğinde tahminleri
# 0-100 aralığında tutmak için kullanılacak.
import numpy as np

# Path:
# Dosya yollarını daha güvenli ve işletim sisteminden
# bağımsız şekilde yönetmemizi sağlar.
from pathlib import Path


# --------------------------------------------------
# Proje ana klasörünü bulma
# --------------------------------------------------

# Path.cwd():
# Notebook'un şu anda hangi klasörde çalıştığını verir.
#
# Day 11 notebook'u notebooks/ klasöründe olduğu için
# current_dir büyük ihtimalle:
#
# trend-forecast-project/notebooks
#
# olacaktır.
current_dir = Path.cwd()


# Eğer bulunduğumuz klasörün içinde "src" varsa
# zaten proje ana klasöründeyiz demektir.
if (current_dir / "src").exists():

    project_root = current_dir


# Eğer bir üst klasörde "src" varsa,
# notebook notebooks/ içinden çalışıyor demektir.
elif (current_dir.parent / "src").exists():

    project_root = current_dir.parent


# Hiçbiri bulunamazsa yanlış klasörde çalışıyoruz.
else:

    raise FileNotFoundError(
        "Proje ana klasörü bulunamadı."
    )


print(
    "Project root:",
    project_root,
)

Project root: /Users/nihalapple/Desktop/trend-forecast-project


In [2]:
# --------------------------------------------------
# Güncellenmiş Google Trends verisini yükleme
# --------------------------------------------------

# Day 9'da oluşturduğumuz ve son gözlemi
# 2026-08-09 olan güncel veri setinin yolu.
data_path = (
    project_root
    / "data"
    / "processed"
    / "google_trends_ai_3y_updated_2026-08-09.csv"
)


# parse_dates=["date"]:
# date sütununu metin yerine gerçek datetime
# veri tipine dönüştürür.
#
# index_col="date":
# date sütununu DataFrame'in zaman index'i yapar.
updated_data = pd.read_csv(
    data_path,
    parse_dates=["date"],
    index_col="date",
)


# --------------------------------------------------
# Verinin doğru yüklendiğini kontrol etme
# --------------------------------------------------

print(
    "Satır sayısı:",
    len(updated_data),
)

print(
    "İlk tarih:",
    updated_data.index.min(),
)

print(
    "Son tarih:",
    updated_data.index.max(),
)


# Son 5 haftayı kontrol ediyoruz.
display(
    updated_data.tail()
)

Satır sayısı: 159
İlk tarih: 2023-07-30 00:00:00
Son tarih: 2026-08-09 00:00:00


,chatgpt,gemini,claude
date,,,
2026-07-12,65,37,15
2026-07-19,69,38,16
2026-07-26,66,38,16
2026-08-02,66,40,14
2026-08-09,70,40,15


## Final Model Konfigürasyonu

Güncel veri 2026-08-09 tarihine kadar olan 159 haftalık Google Trends
gözlemlerini içermektedir.

Önceki model karşılaştırmaları ve güncellenmiş time-series cross-validation
sonuçlarına göre final model seçimleri:

- **ChatGPT:** Prophet + XGBoost Ensemble
- **Gemini:** Naive
- **Claude:** Naive

ChatGPT Ensemble modelinde Prophet ve XGBoost tahminleri %50-%50
ağırlıkla birleştirilmektedir.

XGBoost final parametreleri:

- `n_lags = 8`
- `n_estimators = 300`
- `max_depth = 2`
- `learning_rate = 0.03`
- `random_state = 42`

XGBoost feature seti:

- `lag_1` ... `lag_8`
- `change_1`
- `change_2`

Prophet için:

- `changepoint_prior_scale = 1.0`
- `yearly_seasonality = "auto"`

Model eğitim kodlarının notebook içerisinde tekrar edilmesini önlemek amacıyla
`src/forecasting.py` içerisine `train_prophet_model()` ve
`train_xgb_model()` fonksiyonları eklendi.

Böylece model değerlendirme, model eğitme ve gelecek tahmini üretme
sorumlulukları daha modüler hale getirildi.

In [4]:
# --------------------------------------------------
# Final model seçimleri
# --------------------------------------------------

# Day 9'daki güncel bounded cross-validation
# sonuçlarına göre seçilen modelleri burada
# açık biçimde saklıyoruz.
#
# Böylece ileride hangi trend için hangi modelin
# kullanıldığını kodun farklı yerlerinde aramak
# zorunda kalmayacağız.
FINAL_MODEL_SELECTION = {
    "chatgpt": "ensemble",
    "gemini": "naive",
    "claude": "naive",
}


# ChatGPT Ensemble içinde kullanılan ağırlıklar.
#
# 0.5 Prophet + 0.5 XGBoost
CHATGPT_PROPHET_WEIGHT = 0.5
CHATGPT_XGB_WEIGHT = 0.5


# XGBoost için daha önce tuning sonucunda
# seçtiğimiz final parametreler.
XGB_PARAMS = {
    "n_estimators": 300,
    "max_depth": 2,
    "learning_rate": 0.03,
    "random_state": 42,
}


# XGBoost geçmişten 8 haftalık lag kullanıyor.
N_LAGS = 8


display(
    FINAL_MODEL_SELECTION
)

print(
    "XGBoost params:",
    XGB_PARAMS,
)

{'chatgpt': 'ensemble', 'gemini': 'naive', 'claude': 'naive'}

XGBoost params: {'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.03, 'random_state': 42}


### Final Model Nesnelerinin Oluşturulması

Cross-validation ve model karşılaştırmaları tamamlandıktan sonra seçilen modeller
tüm güncel veri kullanılarak yeniden eğitilecektir.

Güncel model seçimleri:

- ChatGPT → Prophet + XGBoost Ensemble
- Gemini → Naive
- Claude → Naive

ChatGPT Ensemble iki ayrı eğitilmiş modelden oluşmaktadır:

- Prophet
- XGBoost

Bu nedenle iki model ayrı ayrı kaydedilecek ve ensemble ağırlıkları metadata
içerisinde saklanacaktır.

Prophet modeli kendi JSON serialization yöntemiyle, XGBoost modeli ise XGBoost'un
model kaydetme yöntemiyle JSON formatında saklanacaktır.

Gemini ve Claude için kullanılan Naive yaklaşımında eğitilmiş karmaşık bir model
nesnesi bulunmadığından, son gözlenen değerler ve model bilgileri metadata
içerisinde tutulacaktır.

In [5]:
# --------------------------------------------------
# Final modeller için gerekli importlar
# --------------------------------------------------

# Prophet:
# ChatGPT Ensemble'ın istatistiksel model parçası.
from prophet import Prophet


# Prophet'in resmi model kaydetme yöntemi.
#
# model_to_json():
# Eğitilmiş Prophet modelini JSON metnine dönüştürür.
#
# Daha sonra model_from_json() ile tekrar yüklenebilir.
from prophet.serialize import (
    model_to_json,
    model_from_json,
)


# XGBRegressor:
# ChatGPT Ensemble'ın makine öğrenmesi parçası.
from xgboost import XGBRegressor


# json:
# Model metadata bilgilerini JSON dosyasına
# kaydetmek için Python'un standart kütüphanesi.
import json

/Users/nihalapple/Desktop/trend-forecast-project/cotale-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [6]:
# --------------------------------------------------
# Models klasörünü hazırlama
# --------------------------------------------------

# Final eğitilmiş modelleri burada saklayacağız.
models_dir = (
    project_root
    / "models"
)


# mkdir():
# Klasörü oluşturur.
#
# parents=True:
# Üst klasörlerden biri eksikse onu da oluşturabilir.
#
# exist_ok=True:
# models klasörü zaten varsa hata vermez.
models_dir.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Models directory:",
    models_dir,
)

Models directory: /Users/nihalapple/Desktop/trend-forecast-project/models


Cross-validation sırasında
→ geçmişin bir kısmıyla eğit
→ sonraki 4 haftayı test et

Final modelde
→ elimizdeki tüm 159 haftayı kullan
→ gerçek geleceği tahmin et

In [7]:
# --------------------------------------------------
# forecasting.py dosyasındaki yeni değişiklikleri yükleme
# --------------------------------------------------

import sys
import importlib


# project_root'u Day 11'in başında oluşturmuştuk.
#
# Python'un src paketini bulabilmesi için
# proje ana klasörünü import yollarına ekliyoruz.
#
# Eğer zaten ekliyse tekrar eklemiyoruz.
if str(project_root) not in sys.path:
    sys.path.append(
        str(project_root)
    )


# forecasting modülünü import ediyoruz.
import src.forecasting


# importlib.reload():
#
# forecasting.py dosyasını daha önce notebook içinde
# import etmiş olsak bile, dosyaya az önce yaptığımız
# yeni değişiklikleri tekrar belleğe yükler.
importlib.reload(
    src.forecasting
)


# Yeni oluşturduğumuz fonksiyonları
# doğrudan notebook içinde kullanabilmek için import ediyoruz.
from src.forecasting import (
    train_prophet_model,
    train_xgb_model,
)


print(
    "forecasting.py başarıyla yeniden yüklendi."
)

forecasting.py başarıyla yeniden yüklendi.


## Model Persistence

Final Prophet ve XGBoost modelleri güncel verinin tamamı kullanılarak
eğitildi ve `models/` klasörüne kaydedildi.

Oluşturulan model dosyaları:

- `chatgpt_prophet_as_of_2026-08-09.json`
- `chatgpt_xgb_as_of_2026-08-09.json`

Prophet modeli kendi JSON serialization yöntemiyle, XGBoost modeli ise
XGBoost'un model kaydetme yöntemiyle saklandı.

Model dosyalarının yalnızca oluşturulmuş olması yeterli kabul edilmedi.
Her iki model de diskten tekrar yüklenerek aynı girdiler üzerinde
orijinal modellerle karşılaştırıldı.

Sonuç:

- Prophet save/load tahminleri: **Identical = True**
- XGBoost save/load tahminleri: **Identical = True**

Böylece kaydedilmiş modellerin daha sonra yeniden eğitim yapılmadan
yüklenip kullanılabildiği doğrulandı.

In [8]:
# --------------------------------------------------
# Yeni training fonksiyonlarını kontrol etme
# --------------------------------------------------

print(
    "train_prophet_model:",
    train_prophet_model
)

print(
    "train_xgb_model:",
    train_xgb_model
)

train_prophet_model: <function train_prophet_model at 0x11cdccca0>
train_xgb_model: <function train_xgb_model at 0x11cdcd170>


### Final Modellerin Tüm Güncel Veri Üzerinde Eğitilmesi

Model değerlendirme ve seçim aşamaları tamamlandıktan sonra final modeller,
mevcut tüm güncel veri kullanılarak yeniden eğitilecektir.

ChatGPT için seçilen yöntem Prophet ve XGBoost modellerinin birleşiminden oluşan
Ensemble modelidir.

Bu nedenle ChatGPT için:

- final Prophet modeli,
- final XGBoost modeli

ayrı ayrı eğitilecektir.

Model eğitim işlemleri notebook içerisinde tekrar yazılmayacak, `src/forecasting.py`
içerisinde oluşturulan tekrar kullanılabilir training fonksiyonları kullanılacaktır.

In [9]:
# --------------------------------------------------
# ChatGPT final modellerini eğitme
# --------------------------------------------------

# Güncel ChatGPT zaman serisini alıyoruz.
#
# .copy():
# Orijinal updated_data DataFrame'ini değiştirmeden
# ayrı bir Series üzerinde çalışmamızı sağlar.
chatgpt_series = (
    updated_data["chatgpt"]
    .copy()
)


# --------------------------------------------------
# Final Prophet modeli
# --------------------------------------------------

# train_prophet_model():
# forecasting.py içerisinde az önce oluşturduğumuz
# tekrar kullanılabilir training fonksiyonudur.
#
# Fonksiyon:
# 1. Series'i Prophet'in ds-y formatına çevirir.
# 2. Prophet modelini oluşturur.
# 3. Tüm seri üzerinde fit eder.
# 4. Eğitilmiş model nesnesini döndürür.
chatgpt_prophet_model = train_prophet_model(
    series=chatgpt_series,

    # Day 5-8 değerlendirmelerinde kullandığımız
    # final trend esnekliği ayarı.
    changepoint_prior_scale=1.0,

    # Daha önce standartlaştırdığımız Prophet ayarı.
    yearly_seasonality="auto",
)


print(
    "ChatGPT Prophet modeli eğitildi."
)


# --------------------------------------------------
# Final XGBoost modeli
# --------------------------------------------------

# train_xgb_model():
# lag_1 ... lag_8 ile change_1 ve change_2
# feature'larını kendi içerisinde oluşturur.
#
# Daha sonra XGBoost modelini tüm mevcut
# training örnekleri üzerinde eğitir.
chatgpt_xgb_model = train_xgb_model(
    series=chatgpt_series,

    # Geçmiş 8 haftalık bilgiyi kullanıyoruz.
    n_lags=8,

    # Day 6 tuning sonucunda seçilen parametreler.
    max_depth=2,
    n_estimators=300,
    learning_rate=0.03,
    random_state=42,
)


print(
    "ChatGPT XGBoost modeli eğitildi."
)

11:49:17 - cmdstanpy - INFO - Chain [1] start processing
11:49:17 - cmdstanpy - INFO - Chain [1] done processing


ChatGPT Prophet modeli eğitildi.
ChatGPT XGBoost modeli eğitildi.


In [10]:
# --------------------------------------------------
# Models klasörünü hazırlama
# --------------------------------------------------

# Final modelleri proje içerisindeki
# models/ klasöründe saklayacağız.
models_dir = (
    project_root
    / "models"
)


# Klasör zaten varsa hata vermez.
# Yoksa oluşturur.
models_dir.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "Models klasörü:",
    models_dir
)

Models klasörü: /Users/nihalapple/Desktop/trend-forecast-project/models


In [11]:
# --------------------------------------------------
# Prophet model serialization importu
# --------------------------------------------------

# model_to_json:
# Eğitilmiş Prophet modelini JSON metnine dönüştürür.
#
# model_from_json:
# Daha sonra JSON dosyasından modeli geri yükler.
from prophet.serialize import (
    model_to_json,
    model_from_json,
)

In [12]:
# --------------------------------------------------
# Final Prophet modelini kaydetme
# --------------------------------------------------

# Model dosyasının adında veri kesim tarihini tutuyoruz.
#
# Böylece ileride modelin hangi tarihe kadar olan
# veriyle eğitildiğini kolayca anlayabiliriz.
prophet_model_path = (
    models_dir
    / "chatgpt_prophet_as_of_2026-08-09.json"
)


# Dosyayı yazma modunda açıyoruz.
with open(
    prophet_model_path,
    "w",
    encoding="utf-8",
) as file:

    # Eğitilmiş Prophet nesnesini JSON metnine
    # çevirip dosyaya yazıyoruz.
    file.write(
        model_to_json(
            chatgpt_prophet_model
        )
    )


print(
    "Prophet modeli kaydedildi:",
    prophet_model_path,
)

Prophet modeli kaydedildi: /Users/nihalapple/Desktop/trend-forecast-project/models/chatgpt_prophet_as_of_2026-08-09.json


In [13]:
# --------------------------------------------------
# Final XGBoost modelini kaydetme
# --------------------------------------------------

xgb_model_path = (
    models_dir
    / "chatgpt_xgb_as_of_2026-08-09.json"
)


# save_model():
# Eğitilmiş XGBoost modelinin öğrenilmiş
# ağaç yapısını dosyaya kaydeder.
chatgpt_xgb_model.save_model(
    xgb_model_path
)


print(
    "XGBoost modeli kaydedildi:",
    xgb_model_path,
)

XGBoost modeli kaydedildi: /Users/nihalapple/Desktop/trend-forecast-project/models/chatgpt_xgb_as_of_2026-08-09.json


### Kaydedilen Modellerin Geri Yüklenmesi ve Doğrulanması

Final Prophet ve XGBoost modelleri `models/` klasörüne kaydedildikten sonra
dosyalardan tekrar yüklenecektir.

Model dosyasının başarıyla oluşturulması tek başına yeterli değildir.
Kaydedilen modelin daha sonra tekrar yüklenebildiği ve eğitilmiş orijinal modelle
aynı tahminleri üretebildiği doğrulanmalıdır.

Bu nedenle:

1. Prophet modeli JSON dosyasından yeniden yüklenecek.
2. XGBoost modeli JSON dosyasından yeniden yüklenecek.
3. Orijinal ve yeniden yüklenen modeller aynı gelecek girdileri üzerinde çalıştırılacak.
4. Tahminlerin aynı olup olmadığı karşılaştırılacaktır.

Bu işlem model persistence doğrulaması olarak kullanılacaktır.

In [14]:
# --------------------------------------------------
# Prophet modelini JSON dosyasından geri yükleme
# --------------------------------------------------

# prophet_model_path:
# Bir önceki adımda kaydettiğimiz Prophet JSON dosyasının yolu.
#
# Dosyayı bu kez "r" yani read modunda açıyoruz.
with open(
    prophet_model_path,
    "r",
    encoding="utf-8",
) as file:

    # file.read():
    # JSON dosyasının içeriğini metin olarak okur.
    prophet_json = file.read()


# model_from_json():
# JSON metnini tekrar kullanılabilir
# Prophet model nesnesine dönüştürür.
loaded_prophet_model = model_from_json(
    prophet_json
)


print(
    "Prophet modeli başarıyla geri yüklendi."
)

Prophet modeli başarıyla geri yüklendi.


In [15]:
# --------------------------------------------------
# Prophet persistence testi
# --------------------------------------------------

# make_future_dataframe():
# Eğitilmiş modelin son tarihinden sonra
# 4 yeni haftalık tarih oluşturur.
#
# periods=4:
# 4 haftalık gelecek tahmini.
#
# freq="W-SUN":
# Veri setimiz Pazar tarihli haftalık gözlemlerden oluşuyor.
prophet_future = (
    chatgpt_prophet_model
    .make_future_dataframe(
        periods=4,
        freq="W-SUN",
    )
)


# --------------------------------------------------
# Orijinal model tahmini
# --------------------------------------------------

original_prophet_forecast = (
    chatgpt_prophet_model
    .predict(
        prophet_future
    )
    .tail(4)
)


# --------------------------------------------------
# Dosyadan yüklenen model tahmini
# --------------------------------------------------

loaded_prophet_forecast = (
    loaded_prophet_model
    .predict(
        prophet_future
    )
    .tail(4)
)


# İki modelin tahminlerini tek tabloda karşılaştırıyoruz.
prophet_persistence_test = pd.DataFrame(
    {
        "Date": original_prophet_forecast["ds"].values,

        "Original_Prophet": (
            original_prophet_forecast["yhat"].values
        ),

        "Loaded_Prophet": (
            loaded_prophet_forecast["yhat"].values
        ),
    }
)


# İki tahmin arasındaki mutlak farkı hesaplıyoruz.
#
# Başarılı bir serialization/deserialization işleminde
# farkın 0 veya floating-point seviyesinde çok küçük
# olması beklenir.
prophet_persistence_test["Difference"] = (
    prophet_persistence_test["Original_Prophet"]
    - prophet_persistence_test["Loaded_Prophet"]
).abs()


display(
    prophet_persistence_test
)


# np.allclose():
# İki sayısal dizinin pratik olarak aynı olup
# olmadığını kontrol eder.
#
# Floating-point hesaplamalarda doğrudan == yerine
# bunu kullanmak daha güvenlidir.
prophet_same = np.allclose(
    prophet_persistence_test["Original_Prophet"],
    prophet_persistence_test["Loaded_Prophet"],
)


print(
    "Prophet predictions identical:",
    prophet_same,
)

,Date,Original_Prophet,Loaded_Prophet,Difference
0,2026-08-16,68.676012,68.676012,0.0
1,2026-08-23,70.363693,70.363693,0.0
2,2026-08-30,72.009915,72.009915,0.0
3,2026-09-06,73.308497,73.308497,0.0


Prophet predictions identical: True


In [16]:
# --------------------------------------------------
# XGBoost modelini JSON dosyasından geri yükleme
# --------------------------------------------------

from xgboost import XGBRegressor


# Henüz eğitilmemiş boş bir XGBoost model nesnesi.
loaded_xgb_model = XGBRegressor()


# load_model():
# models/ klasöründeki kaydedilmiş ağaç yapısını
# bu model nesnesinin içine yükler.
loaded_xgb_model.load_model(
    str(xgb_model_path)
)


print(
    "XGBoost modeli başarıyla geri yüklendi."
)

XGBoost modeli başarıyla geri yüklendi.


In [17]:
# --------------------------------------------------
# XGBoost persistence testi için gelecek feature'ı
# --------------------------------------------------

# Son 8 haftalık gerçek ChatGPT değerini alıyoruz.
#
# [::-1]:
# sırayı ters çevirir.
#
# Böylece:
#
# lags[0] -> lag_1 -> en son hafta
# lags[1] -> lag_2
# ...
# lags[7] -> lag_8
xgb_lags = (
    chatgpt_series
    .iloc[-8:]
    .astype(float)
    .tolist()[::-1]
)


# Modelin eğitim sırasında kullandığı
# feature isimlerini aynı sırayla oluşturuyoruz.
xgb_feature_columns = [
    *[
        f"lag_{i}"
        for i in range(
            1,
            9,
        )
    ],
    "change_1",
    "change_2",
]


# --------------------------------------------------
# Lag feature'larını dictionary haline getirme
# --------------------------------------------------

# enumerate(xgb_lags):
# listedeki değerleri sıra numarasıyla gezer.
#
# i=0 -> lag_1
# i=1 -> lag_2
# ...
xgb_future_features = {
    f"lag_{i + 1}": value
    for i, value in enumerate(
        xgb_lags
    )
}


# Son iki haftanın değişimini hesaplıyoruz.
xgb_future_features["change_1"] = (
    xgb_lags[0]
    - xgb_lags[1]
)


# Ondan önceki değişimi hesaplıyoruz.
xgb_future_features["change_2"] = (
    xgb_lags[1]
    - xgb_lags[2]
)


# XGBoost predict() fonksiyonu DataFrame beklediği için
# dictionary'yi tek satırlık DataFrame'e dönüştürüyoruz.
X_next_week = pd.DataFrame(
    [
        xgb_future_features
    ],
    columns=xgb_feature_columns,
)


display(
    X_next_week
)

,lag_1,lag_2,lag_3,lag_4,lag_5,lag_6,lag_7,lag_8,change_1,change_2
0,70.0,66.0,66.0,69.0,65.0,65.0,65.0,68.0,4.0,0.0


In [18]:
# --------------------------------------------------
# XGBoost orijinal ve loaded model karşılaştırması
# --------------------------------------------------

# Bellekteki orijinal eğitilmiş modelin tahmini.
original_xgb_prediction = (
    chatgpt_xgb_model
    .predict(
        X_next_week
    )[0]
)


# JSON'dan geri yüklediğimiz modelin tahmini.
loaded_xgb_prediction = (
    loaded_xgb_model
    .predict(
        X_next_week
    )[0]
)


# İki tahmin arasındaki fark.
xgb_difference = abs(
    original_xgb_prediction
    - loaded_xgb_prediction
)


print(
    "Original XGBoost prediction:",
    original_xgb_prediction,
)

print(
    "Loaded XGBoost prediction:",
    loaded_xgb_prediction,
)

print(
    "Difference:",
    xgb_difference,
)


# np.isclose():
# İki tek sayının pratik olarak aynı olup
# olmadığını kontrol eder.
xgb_same = np.isclose(
    original_xgb_prediction,
    loaded_xgb_prediction,
)


print(
    "XGBoost predictions identical:",
    xgb_same,
)

Original XGBoost prediction: 71.82043
Loaded XGBoost prediction: 71.82043
Difference: 0.0
XGBoost predictions identical: True


## Model Metadata

Model dosyalarının yanında model konfigürasyonunu açıklayan ayrı bir
metadata dosyası oluşturuldu:

`model_metadata_as_of_2026-08-09.json`

Metadata modelin kendisi değildir; model hakkında bilgi taşıyan üst veridir.

Metadata içerisinde:

- veri kesim tarihi,
- forecast horizon,
- trend bazında seçilen final modeller,
- Ensemble ağırlıkları,
- Prophet parametreleri,
- XGBoost parametreleri,
- kullanılan XGBoost feature'ları,
- model dosyalarının isimleri,
- persistence test sonuçları,
- Gemini ve Claude için gerekli son gözlenen değerler

saklanmaktadır.

Bu yapı model konfigürasyonunun daha sonra anlaşılmasını ve yeniden
üretilebilirliğini kolaylaştırmaktadır.

### Final Model Metadata Bilgilerinin Kaydedilmesi

Eğitilmiş Prophet ve XGBoost modellerinin başarıyla kaydedilip yeniden
yüklenebildiği doğrulandı.

Model dosyalarının yanında, modellerin nasıl oluşturulduğunu açıklayan bir
metadata dosyası da saklanacaktır.

Metadata içerisinde:

- kullanılan veri kesim tarihi,
- her trend için seçilen model,
- ChatGPT Ensemble ağırlıkları,
- Prophet parametreleri,
- XGBoost parametreleri,
- XGBoost feature bilgileri,
- Gemini ve Claude için Naive modelin kullandığı son değerler

saklanacaktır.

Metadata dosyası modelin kendisini değil, modelin nasıl oluşturulduğunu ve
hangi konfigürasyonun final olarak seçildiğini açıklamaktadır.

In [19]:
# --------------------------------------------------
# Final model metadata oluşturma
# --------------------------------------------------

# Veri setinin en son gözlem tarihini alıyoruz.
#
# strftime("%Y-%m-%d"):
# Timestamp değerini örneğin
#
# 2026-08-09
#
# formatında okunabilir bir metne dönüştürür.
data_cutoff_date = (
    updated_data.index[-1]
    .strftime("%Y-%m-%d")
)


# --------------------------------------------------
# Gemini ve Claude Naive değerleri
# --------------------------------------------------

# Naive forecasting:
# gelecekteki değerin son gözlenen değerle
# aynı kalacağını varsayar.
#
# Bu nedenle ayrıca eğitilmiş bir model nesnesi
# kaydetmemize gerek yoktur.
#
# Sadece serinin son değerini bilmemiz yeterlidir.
gemini_last_value = float(
    updated_data["gemini"].iloc[-1]
)

claude_last_value = float(
    updated_data["claude"].iloc[-1]
)


# --------------------------------------------------
# Metadata dictionary
# --------------------------------------------------

# Bu dictionary final forecasting sisteminin
# hangi konfigürasyonla çalıştığını açıklayacak.
model_metadata = {

    # Metadata'nın hangi veri tarihine kadar
    # olan bilgilerle oluşturulduğunu gösterir.
    "data_cutoff_date": data_cutoff_date,

    # Forecast horizon:
    # mevcut proje hedefimiz 4 haftalık tahmin.
    "forecast_horizon_weeks": 4,

    # Google Trends değerlerinin doğal sınırı.
    "prediction_range": [
        0,
        100,
    ],

    # --------------------------------------------------
    # ChatGPT
    # --------------------------------------------------
    "chatgpt": {

        # Cross-validation sonucunda seçilen
        # final yaklaşım.
        "selected_model": "ensemble",

        # Ensemble iki modelden oluşuyor.
        "components": [
            "prophet",
            "xgboost",
        ],

        # Prophet ve XGBoost eşit ağırlıklı.
        "weights": {
            "prophet": 0.5,
            "xgboost": 0.5,
        },

        # Prophet final parametreleri.
        "prophet": {
            "changepoint_prior_scale": 1.0,
            "yearly_seasonality": "auto",
            "weekly_seasonality": False,
            "daily_seasonality": False,

            # Kaydedilen model artifact'ının adı.
            "model_file": (
                prophet_model_path.name
            ),
        },

        # XGBoost final parametreleri.
        "xgboost": {
            "n_lags": 8,
            "n_estimators": 300,
            "max_depth": 2,
            "learning_rate": 0.03,
            "random_state": 42,

            # Modelin kullandığı feature'lar.
            "features": [
                "lag_1",
                "lag_2",
                "lag_3",
                "lag_4",
                "lag_5",
                "lag_6",
                "lag_7",
                "lag_8",
                "change_1",
                "change_2",
            ],

            # Kaydedilen XGBoost artifact'ının adı.
            "model_file": (
                xgb_model_path.name
            ),
        },

        # Save-load doğrulamasının geçtiğini de
        # metadata içinde kayıt altına alıyoruz.
        "persistence_test": {
            "prophet_identical": bool(
                prophet_same
            ),
            "xgboost_identical": bool(
                xgb_same
            ),
        },
    },

    # --------------------------------------------------
    # Gemini
    # --------------------------------------------------
    "gemini": {

        # Güncel cross-validation sonucunda
        # final model olarak Naive seçildi.
        "selected_model": "naive",

        # Naive modelin kullanacağı son gerçek değer.
        "last_observed_value": gemini_last_value,
    },

    # --------------------------------------------------
    # Claude
    # --------------------------------------------------
    "claude": {

        # Claude için de final model Naive.
        "selected_model": "naive",

        # Naive tahmin için gerekli son değer.
        "last_observed_value": claude_last_value,
    },
}


# Dictionary'nin içeriğini notebook'ta
# okunabilir biçimde kontrol ediyoruz.
model_metadata

{'data_cutoff_date': '2026-08-09',
 'forecast_horizon_weeks': 4,
 'prediction_range': [0, 100],
 'chatgpt': {'selected_model': 'ensemble',
  'components': ['prophet', 'xgboost'],
  'weights': {'prophet': 0.5, 'xgboost': 0.5},
  'prophet': {'changepoint_prior_scale': 1.0,
   'yearly_seasonality': 'auto',
   'weekly_seasonality': False,
   'daily_seasonality': False,
   'model_file': 'chatgpt_prophet_as_of_2026-08-09.json'},
  'xgboost': {'n_lags': 8,
   'n_estimators': 300,
   'max_depth': 2,
   'learning_rate': 0.03,
   'random_state': 42,
   'features': ['lag_1',
    'lag_2',
    'lag_3',
    'lag_4',
    'lag_5',
    'lag_6',
    'lag_7',
    'lag_8',
    'change_1',
    'change_2'],
   'model_file': 'chatgpt_xgb_as_of_2026-08-09.json'},
  'persistence_test': {'prophet_identical': True, 'xgboost_identical': True}},
 'gemini': {'selected_model': 'naive', 'last_observed_value': 40.0},
 'claude': {'selected_model': 'naive', 'last_observed_value': 15.0}}

In [20]:
# --------------------------------------------------
# Metadata JSON dosyasını kaydetme
# --------------------------------------------------

metadata_path = (
    models_dir
    / "model_metadata_as_of_2026-08-09.json"
)


# JSON dosyasını yazma modunda açıyoruz.
with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as file:

    # json.dump():
    # Python dictionary'sini JSON dosyasına yazar.
    #
    # indent=4:
    # JSON'un tek satır yerine girintili ve
    # insanlar tarafından okunabilir olmasını sağlar.
    #
    # ensure_ascii=False:
    # Türkçe karakterlerin okunabilir şekilde
    # yazılmasını sağlar.
    json.dump(
        model_metadata,
        file,
        indent=4,
        ensure_ascii=False,
    )


print(
    "Metadata kaydedildi:",
    metadata_path,
)

Metadata kaydedildi: /Users/nihalapple/Desktop/trend-forecast-project/models/model_metadata_as_of_2026-08-09.json


In [21]:
# --------------------------------------------------
# Models klasörü kontrolü
# --------------------------------------------------

# iterDir():
# models/ klasöründeki dosyaları tek tek verir.
#
# sorted():
# isimlerine göre sıralar.
for model_file in sorted(
    models_dir.iterdir()
):

    print(
        model_file.name
    )

.gitkeep
chatgpt_prophet_as_of_2026-08-09.json
chatgpt_xgb_as_of_2026-08-09.json
model_metadata_as_of_2026-08-09.json


Burada özellikle Gemini ve Claude için ayrı .json model dosyası olmaması hata değil. Naive modelin öğrenilmiş parametreleri/ağaçları yok; sadece son gözlenen değeri kullanıyor. O yüzden onlar metadata içinde temsil ediliyor.

### Metadata Nedir?

Tabii. Önce bunu netleştirelim; sonra kaldığımız yerden devam ederiz.

**JSON**, “JavaScript Object Notation” demek. Temel amacı, **yapılandırılmış bilgiyi hem insanların okuyabileceği hem de bilgisayarların kolayca işleyebileceği bir metin formatında saklamak ve sistemler arasında taşımak**. JSON; isim-değer çiftleri ve listeler gibi basit yapılar kullanıyor ve bugün programlama dilinden bağımsız bir veri değişim formatı olarak standartlaştırılmış durumda. ([Ecma International][1])

Mesela bizim Python’daki şu dictionary:

```python
model_info = {
    "model": "XGBoost",
    "n_lags": 8,
    "learning_rate": 0.03
}
```

JSON dosyasında neredeyse aynı görünür:

```json
{
    "model": "XGBoost",
    "n_lags": 8,
    "learning_rate": 0.03
}
```

Bu yüzden JSON çok kullanışlı: Python okuyabilir, Java okuyabilir, JavaScript okuyabilir, bir web uygulaması okuyabilir. Yani örneğin ileride Streamlit dashboard’umuz bu bilgileri okuyabilir.

### JSON'u kim buldu?

Burada küçük ama ilginç bir ayrıntı var. JSON en çok **Douglas Crockford** ile ilişkilendiriliyor. Crockford JSON’u yaygınlaştırdı, adlandırdı, belgeledi ve standartlaştırılmasında önemli rol oynadı. Fakat kendisi “JSON’u icat ettim” demekten ziyade **“keşfettim”** demeyi tercih ediyor; çünkü JavaScript’in sözdiziminde taşınabilir veri yapılarını temsil etmeye uygun olan bu yapı zaten vardı. Crockford, bunu özellikle 2000’lerin başında web uygulamalarında veri alışverişi için kullanışlı bir format olarak ortaya çıkardı. ([crockford.com][2])

Yani kabaca:

```text
JavaScript'te nesne yazma biçimi zaten vardı
               ↓
Douglas Crockford bunun
veri alışverişi için çok uygun
olduğunu fark etti
               ↓
JSON adıyla tanımlandı/yaygınlaştırıldı
               ↓
Web sistemlerinde çok yaygınlaştı
               ↓
Daha sonra standartlaştırıldı
```

Bugün JSON’un sözdizimi ECMA-404 standardında tanımlanıyor. ([Ecma International][1])

### Peki neden böyle bir şeye ihtiyaç vardı?

Bir düşün: Python programın bir web sunucusuna şunu gönderecek:

> Kullanıcının adı Nihal, yaşı 20, aktif mi evet, kullandığı modeller Prophet ve XGBoost.

Bunu düz metin olarak:

```text
Nihal 20 evet Prophet XGBoost
```

yazarsak bilgisayar açısından belirsiz. Hangisi isim? Hangisi yaş? Kaç tane model var?

JSON ise yapıyı açıkça belirtiyor:

```json
{
    "name": "Nihal",
    "age": 20,
    "active": true,
    "models": [
        "Prophet",
        "XGBoost"
    ]
}
```

Artık bilgisayar şunu biliyor:

```text
name    → Nihal
age     → 20
active  → true
models  → liste
```

JSON’un temel gücü bu.

---

## Metadata nedir?

`metadata` kelimesini parçalarsak:

**meta + data → data hakkında data**

Türkçede genellikle **“üst veri”** deniyor.

Mesela elimizde bir fotoğraf olsun:

```text
fotoğrafın kendisi → DATA
```

Ama:

```text
çekildiği tarih
kamera modeli
dosya boyutu
çözünürlük
```

fotoğrafın kendisi değil; **fotoğraf hakkında bilgiler**.

Bunlar metadata.

Bizim projeye gelirsek:

### Modelin kendisi:

```text
chatgpt_xgb_as_of_2026-08-09.json
```

içinde XGBoost’un öğrendiği ağaçlar ve model yapısı bulunuyor.

Bu **model artifact’ı**, yani asıl model.

Ama model hakkında şunları da bilmek istiyoruz:

```text
Bu hangi model?
Hangi tarihe kadar veriyle eğitildi?
Kaç lag kullandı?
learning_rate neydi?
max_depth neydi?
Hangi feature'lar vardı?
ChatGPT için hangi model seçildi?
Ensemble ağırlıkları neydi?
```

Bunlar modelin **metadata’sı**.

Bizim oluşturduğumuz:

```text
model_metadata_as_of_2026-08-09.json
```

dosyasının amacı tam olarak bu.

Mesela:

```json
{
    "data_cutoff_date": "2026-08-09",

    "chatgpt": {
        "selected_model": "ensemble",

        "weights": {
            "prophet": 0.5,
            "xgboost": 0.5
        },

        "xgboost": {
            "n_lags": 8,
            "n_estimators": 300,
            "max_depth": 2,
            "learning_rate": 0.03
        }
    }
}
```

burada **modelin kendisini saklamıyoruz.**

Şunu saklıyoruz:

> “Bu model hangi koşullarda oluşturuldu?”

### Çok basit benzetme

Bir kavanoz reçel düşün:

```text
Kavanozun içindeki reçel
        ↓
MODEL
```

Kavanozun üzerindeki etiket:

```text
Çilek reçeli
Üretim tarihi: ...
İçindekiler: ...
Gramaj: ...
Parti numarası: ...
```

```
    ↓
```

**METADATA**

Bizim durumda:

```text
chatgpt_xgb...json
       ↓
reçelin kendisi

model_metadata...json
       ↓
kavanozun etiketi
```

gibi.

---

## Peki neden metadata'yı ayrıca tutuyoruz?

Çünkü bugün her şeyi hatırlıyoruz:

> “XGBoost 8 lag kullanıyordu, learning rate 0.03’tü, ChatGPT ensemble’dı...”

Ama bir ay sonra `models/` klasörünü açarsan sadece:

```text
chatgpt_xgb_as_of_2026-08-09.json
```

göreceksin.

“Bu hangi feature’larla eğitilmişti?” diye kodu didiklemek istemeyiz.

Metadata açarsın:

```text
n_lags = 8
features = lag_1 ... lag_8, change_1, change_2
max_depth = 2
n_estimators = 300
learning_rate = 0.03
```

hepsi karşında.

Bu özellikle **reproducibility**, yani “aynı modeli daha sonra tekrar oluşturabilme” açısından önemli.

---

Bir de iki JSON dosyamızın amaçlarının farklı olduğunu ayıralım:

```text
chatgpt_prophet_as_of_2026-08-09.json
chatgpt_xgb_as_of_2026-08-09.json

→ Eğitilmiş MODELLER
```

ve:

```text
model_metadata_as_of_2026-08-09.json

→ Modeller HAKKINDA BİLGİ
```

Hepsinin uzantısı `.json`, ama **aynı şeyi taşımıyorlar**. JSON sadece bilgiyi nasıl yazacağımızı belirleyen format.



### Kaydedilmiş Modellerle Final Forecast Pipeline

Final model dosyaları ve metadata bilgileri kullanılarak yeniden eğitim
gerektirmeden tahmin oluşturabilen bir inference pipeline geliştirildi.

Pipeline:

1. Metadata dosyasını yükler.
2. Veri tarihi ile model tarihini doğrular.
3. Kaydedilmiş Prophet ve XGBoost modellerini yükler.
4. ChatGPT için iki modelin tahminlerini birleştirir.
5. Gemini ve Claude için Naive forecast üretir.
6. Üç trendin 4 haftalık tahminlerini tek DataFrame içerisinde döndürür.

Bu yapı ileride Streamlit dashboard tarafından doğrudan kullanılabilecek
final forecasting katmanının temelini oluşturmaktadır.

In [22]:
# --------------------------------------------------
# pipeline.py dosyasını yükleme
# --------------------------------------------------

import importlib
import src.pipeline


# Dosyaya yeni yazdığımız kodların
# notebook belleğine gelmesini sağlar.
importlib.reload(
    src.pipeline
)


# Final pipeline fonksiyonunu import ediyoruz.
from src.pipeline import (
    generate_final_forecast,
)


print(
    "pipeline.py başarıyla yüklendi."
)

pipeline.py başarıyla yüklendi.


## Final Forecasting Pipeline

Kaydedilmiş modellerin gerçek uygulama ortamında yeniden eğitim yapılmadan
kullanılabilmesi için `src/pipeline.py` modülü oluşturuldu.

Pipeline içerisinde:

- metadata yükleme,
- Prophet modelini diskten yükleme,
- XGBoost modelini diskten yükleme,
- yüklenmiş Prophet modeliyle tahmin üretme,
- yüklenmiş XGBoost modeliyle recursive tahmin üretme,
- Naive forecast üretme,
- ChatGPT Ensemble tahmini oluşturma

işlemleri modüler fonksiyonlara ayrıldı.

`generate_final_forecast()` fonksiyonu tek bir giriş noktası olarak
güncel veri, model dosyaları ve metadata bilgilerini kullanarak ChatGPT,
Gemini ve Claude için 4 haftalık final forecast üretmektedir.

Pipeline ayrıca veri tarihi ile modelin eğitim tarihi arasında uyumsuzluk
olup olmadığını kontrol etmektedir. Böylece eski modelin fark edilmeden
daha yeni veri üzerinde kullanılması engellenmektedir.

In [23]:
# --------------------------------------------------
# Kaydedilmiş modellerle final forecast
# --------------------------------------------------

final_forecast_from_saved_models = (
    generate_final_forecast(

        # Güncel Google Trends verisi
        data=updated_data,

        # Kaydedilmiş modellerin bulunduğu klasör
        models_dir=models_dir,

        # Final metadata dosyası
        metadata_path=metadata_path,
    )
)


display(
    final_forecast_from_saved_models
)

,ChatGPT,Gemini,Claude
Date,,,
2026-08-16,70.248219,40.0,15.0
2026-08-23,70.850414,40.0,15.0
2026-08-30,71.373637,40.0,15.0
2026-09-06,72.022927,40.0,15.0


## Kaydedilmiş Modellerle Final Forecast

Kaydedilmiş Prophet ve XGBoost modelleri kullanılarak, yeniden training
yapılmadan final forecast üretildi.

Elde edilen sonuçlar:

| Tarih | ChatGPT | Gemini | Claude |
|---|---:|---:|---:|
| 2026-08-16 | 70.248219 | 40.0 | 15.0 |
| 2026-08-23 | 70.850414 | 40.0 | 15.0 |
| 2026-08-30 | 71.373637 | 40.0 | 15.0 |
| 2026-09-06 | 72.022927 | 40.0 | 15.0 |

Bu sonuçlar modeller doğrudan eğitildiğinde daha önce elde edilen final
forecast değerleriyle aynı bulundu.

Final tahmin çıktısı:

`reports/final_forecast_as_of_2026-08-09.csv`

dosyasına kaydedildi.

CSV dosyası tekrar okunarak pipeline çıktısıyla karşılaştırıldı ve kayıt
işleminin tahmin değerlerini değiştirmediği doğrulandı.

### Final Forecast Çıktısının Kaydedilmesi

Kaydedilmiş model dosyaları ve metadata kullanılarak final forecasting pipeline
başarıyla çalıştırıldı.

Pipeline tarafından üretilen tahminler, modellerin doğrudan eğitilmesiyle daha
önce elde edilen final tahminlerle aynı sonuçları verdi.

Bu doğrulamanın ardından 4 haftalık final forecast çıktısı `reports/`
klasörüne CSV formatında kaydedilecektir.

Kaydedilen dosya, dashboard ve raporlama aşamalarında kullanılabilecek
standart tahmin çıktısını temsil etmektedir.

In [24]:
# --------------------------------------------------
# Reports klasörünü hazırlama
# --------------------------------------------------

# project_root:
# Projenin ana klasörü.
#
# reports:
# Analiz ve forecasting sonuçlarımızın
# saklandığı klasör.
reports_dir = (
    project_root
    / "reports"
)


# Klasör mevcut değilse oluşturur.
# Zaten varsa hata vermez.
reports_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Final forecast dosya yolu
# --------------------------------------------------

# Dosya adına model/veri kesim tarihini ekliyoruz.
#
# Böylece ileride yeni veri geldiğinde örneğin:
#
# final_forecast_as_of_2026-08-16.csv
#
# şeklinde yeni versiyonlar üretilebilir.
final_forecast_path = (
    reports_dir
    / "final_forecast_as_of_2026-08-09.csv"
)


# --------------------------------------------------
# Forecast'u CSV olarak kaydetme
# --------------------------------------------------

# to_csv():
# DataFrame'i CSV dosyasına yazar.
#
# index=True:
# Date bilgisi DataFrame index'inde olduğu için
# tarihlerin de CSV dosyasına yazılmasını istiyoruz.
final_forecast_from_saved_models.to_csv(
    final_forecast_path,
    index=True,
)


print(
    "Final forecast kaydedildi:",
    final_forecast_path,
)

Final forecast kaydedildi: /Users/nihalapple/Desktop/trend-forecast-project/reports/final_forecast_as_of_2026-08-09.csv


In [25]:
# --------------------------------------------------
# Kaydedilen forecast dosyasını doğrulama
# --------------------------------------------------

# CSV dosyasını yeniden okuyoruz.
#
# parse_dates=["Date"]:
# Date sütununu normal metin yerine
# pandas datetime tipine dönüştürür.
saved_final_forecast = pd.read_csv(
    final_forecast_path,
    parse_dates=["Date"],
    index_col="Date",
)


# Kaydedilmiş dosyanın içeriğini gösteriyoruz.
display(
    saved_final_forecast
)

,ChatGPT,Gemini,Claude
Date,,,
2026-08-16,70.248219,40.0,15.0
2026-08-23,70.850414,40.0,15.0
2026-08-30,71.373637,40.0,15.0
2026-09-06,72.022927,40.0,15.0


In [26]:
# --------------------------------------------------
# Orijinal pipeline çıktısı ile CSV kontrolü
# --------------------------------------------------

# np.allclose():
# İki sayısal tablonun değerlerinin aynı olup
# olmadığını floating-point toleransıyla kontrol eder.
#
# CSV'ye yazma ve tekrar okuma sırasında çok küçük
# ondalık gösterim farkları oluşabileceği için
# doğrudan == yerine allclose kullanıyoruz.
forecast_file_identical = np.allclose(
    final_forecast_from_saved_models.values,
    saved_final_forecast.values,
)


print(
    "Saved forecast identical:",
    forecast_file_identical,
)

Saved forecast identical: True


## Streamlit Dashboard

Final forecasting pipeline hazır hale geldikten sonra model sonuçlarını
teknik olmayan kullanıcıların da inceleyebilmesi amacıyla Streamlit
tabanlı bir dashboard geliştirilmeye başlandı.

Proje ana klasöründe `app.py` oluşturuldu.

Dashboard doğrudan:

- güncel Google Trends verisini,
- `models/` klasöründeki kaydedilmiş modelleri,
- model metadata dosyasını,
- `src/pipeline.py` içerisindeki final forecasting pipeline'ını

kullanmaktadır.

Bu nedenle kullanıcı dashboard üzerinde teknoloji seçtiğinde modellerin
tekrar eğitilmesine gerek kalmadan kaydedilmiş final modeller kullanılarak
tahminler gösterilmektedir.

### Dashboard Özellikleri

Dashboard üzerinde kullanıcı:

- ChatGPT, Gemini veya Claude trendlerinden birini seçebilmektedir.
- Seçilen trendin son Google Trends skorunu görebilmektedir.
- 4 hafta sonrası final tahmin değerini görebilmektedir.
- Kullanılan final modeli görebilmektedir.
- Geçmiş Google Trends verisini interaktif Plotly grafiğinde inceleyebilmektedir.
- Gelecek 4 haftalık final forecast'u aynı grafik üzerinde görebilmektedir.
- Forecast değerlerini tablo halinde inceleyebilmektedir.

Gerçek geçmiş veri ve gelecek forecast grafik üzerinde farklı çizgi
biçimleri kullanılarak birbirinden ayrılmıştır.

Grafik Plotly kullanılarak oluşturulduğu için kullanıcı tarihler ve trend
skorları üzerinde interaktif olarak gezinebilmektedir.

## Anomaly Detection ve Trend Monitoring

Daha önce geliştirilen anomaly detection yaklaşımı notebook seviyesinden
çıkarılarak `src/monitoring.py` içerisine taşındı ve tekrar kullanılabilir
hale getirildi.

Final anomaly detection parametreleri:

- `window = 12`
- `threshold = 3.5`
- `min_absolute_change = 5`

Anomaly detection mevcut haftayı rolling referans hesabına dahil etmemek
için `shift(1)` kullanmaktadır. Böylece bir gözlemin sıra dışı olup olmadığı
yalnızca kendisinden önceki geçmiş davranışa göre değerlendirilmektedir.

Dashboard üzerinde seçilen trend için son gözlemin anomaly durumu ve
anomaly score gösterilmektedir.

Ayrıca geçmişte tespit edilen anomaly noktaları Plotly grafiği üzerinde
ayrı işaretlerle gösterilmiştir. Kullanıcı bu noktaların üzerine geldiğinde
tarih, trend skoru ve anomaly score bilgisini görebilmektedir.

Anomaly detection forecasting modelinin yerine geçmemektedir. Bu katmanın
amacı mevcut veya geçmiş davranıştaki sıra dışı hareketleri izlemektir.

## Early-Warning Mekanizması

Anomaly detection'dan ayrı olarak gelecekteki tahminlere dayalı basit ve
yorumlanabilir bir early-warning mekanizması oluşturuldu.

Bu mekanizma:

1. Mevcut gerçek değer ile forecast horizon sonundaki değer arasındaki
   toplam değişimi,
2. Forecast horizon içerisindeki haftalık değişimlerin ne kadarının
   pozitif olduğunu

kontrol etmektedir.

Başlangıç kuralı olarak:

- toplam artışın en az `5` Google Trends puanı olması,
- haftalık hareketlerin en az `%75`'inin pozitif olması

durumunda "Yükselen Trend Sinyali" üretilmektedir.

Bu eşikler optimize edilmiş nihai istatistiksel parametreler olarak değil,
yorumlanabilir başlangıç kuralları olarak kullanılmaktadır.

Böylece:

- **Anomaly Detection:** "Şu anda sıra dışı bir hareket var mı?"
- **Early Warning:** "Model gelecekte belirgin ve devamlı bir yükseliş
  öngörüyor mu?"

sorularını birbirinden ayrı şekilde ele alan iki monitoring katmanı elde
edildi.